# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The demo client below is deterministic so the notebook can be executed without a model account. For real extraction, omit `model=DemoModel()` to use the package default OpenAI `gpt-4.1-mini` client with `OPENAI_API_KEY`, or inject another adapter implementing `generate(...)`.

In [1]:
from pathlib import Path

from semantic_graphicalizer import SemanticGraphicalizer, load_aesop_fables

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent

stories = load_aesop_fables(limit=2, cache_dir=ROOT / 'data' / 'raw')
[(story.splitlines()[0], len(story)) for story in stories]

[('THE DAW IN BORROWED FEATHERS', 859), ('THE SUN AND THE WIND', 972)]

In [2]:
len(stories), [story.splitlines()[0] for story in stories]

(2, ['THE DAW IN BORROWED FEATHERS', 'THE SUN AND THE WIND'])

## Model adapter

For a real run, replace `DemoModel` with a provider-specific adapter. The core library only requires a `generate(stage=..., prompt=..., schema=..., context=...)` method returning a parsed mapping.

In [3]:
class DemoModel:
    def generate(self, *, stage, prompt, schema, context):
        if stage == 'summarize':
            return {'summary': 'A fable describes characters whose actions lead to a consequence and a moral.'}
        if stage == 'normalize':
            return {'normalized': 'A Character performs an Action that causes a consequence and teaches a Moral.'}
        if stage == 'decompose':
            return {'propositions': [{'id': 'p1', 'text': 'A character performs an action.', 'source_text': 'A fable action.'}, {'id': 'p2', 'text': 'The action teaches a moral.', 'source_text': 'A fable moral.'}]}
        if stage == 'triple':
            prefix = context['chunk_id']
            return {'triples': [
                {'id': 't1', 'proposition_id': f'{prefix}:p1', 'subject': {'mention': 'main character', 'label': 'Character'}, 'predicate': 'performs', 'object': {'mention': 'central action', 'label': 'Action'}, 'proposition': 'A character performs an action.'},
                {'id': 't2', 'proposition_id': f'{prefix}:p2', 'subject': {'mention': 'central action', 'label': 'Action'}, 'predicate': 'teaches', 'object': {'mention': 'moral lesson', 'label': 'Moral'}, 'proposition': 'The action teaches a moral.'},
            ]}
        raise ValueError(stage)

graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
    model=DemoModel(),
)
traces = graphicalizer.fit([stories[0]]).transform_with_trace([stories[0]])
trace = traces[0]
[(p.text, p.proposition_id) for p in trace.propositions]

[('A character performs an action.', 'document-0:chunk-0:p1'),
 ('The action teaches a moral.', 'document-0:chunk-0:p2')]

In [4]:
graphicalizer.display(trace.graph)